<a href="https://colab.research.google.com/github/AhmedMahmoud-123/FlyRank_AI/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os

REPO_URL = 'https://github.com/AhmedMahmoud-123/FlyRank_AI.git'
REPO_DIR = 'FlyRank_AI'

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL}
os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

Working directory: /content/FlyRank_AI


In [2]:
%pip -q install duckdb huggingface_hub

In [3]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [4]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: one modeling row = one content item (content_hash_id) for a client, aggregated across two adjacent 30-day windows. The underlying daily fact is intended to represent content-day observations, but the raw table contains 6,390 duplicate (client, content, date) groups, so uniqueness at that key is not confirmed. The duplicate rows appear to reflect source/availability differences, so the raw daily table should be normalized to one content-day record before calculating window aggregates.

Time window: two adjacent 30-day windows relative to the available performance history:

prev30 = days 31–60 back → feature inputs
last30 = most recent 30 days → label input only

The panel is unbalanced across clients, so available history differs by client. The final modeling table therefore uses the content/client as its unit after daily records have been resolved to a consistent content-day grain.

In [22]:
dupes = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS row_count
    FROM {TABLES['fact_daily']}
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
""").df()

print("Duplicate (client, content, date) groups:", len(dupes))
print("Rows in duplicate groups:", dupes["row_count"].sum())
print("Maximum rows per group:", dupes["row_count"].max())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (client, content, date) groups: 6390
Rows in duplicate groups: 12780
Maximum rows per group: 2


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| **Feature** | `imp_prev30`, `pos_prev30`, `pos_volatility_prev30`, `visible_queries`, `rare_share`, `anon_share`, `top_query_share`
| Knowable before the label window |

| **Label** | `is_declining_label` | The thing I'm predicting |

| **Context** | `client_hash_id`, `content_hash_id` | Grouping / joins only |

| **Excluded** | `imp_last30`, `clk_last30` (label period — leakage), `competition_level` (descriptive metadata), `provider_used`, `model_used` (not used as predictive signals) | Excluded for a specific reason |

In [23]:
# feature_cols must never overlap the label window
feature_cols = ['imp_prev30', 'pos_last30', 'pos_volatility_last30',
                'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
label_inputs = ['imp_last30']
assert not set(feature_cols) & set(label_inputs), "leakage: a feature is also a label input"
print("no overlap between features and label inputs — OK")

no overlap between features and label inputs — OK


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [26]:
# Check missingness in the query-level fields we plan to use
missing = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE content_visible_query_count IS NULL
        ) AS missing_visible_queries,
        COUNT(*) FILTER (
            WHERE rare_query_count IS NULL
        ) AS missing_rare_queries,
        COUNT(*) FILTER (
            WHERE rare_impressions_share IS NULL
        ) AS missing_rare_share,
        COUNT(*) FILTER (
            WHERE anonymized_impressions_share IS NULL
        ) AS missing_anon_share
    FROM {TABLES['fact_query_90d']}
""").df()

print("Query-field missingness:")
display(missing)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query-field missingness:


,total_rows,missing_visible_queries,missing_rare_queries,missing_rare_share,missing_anon_share
0,2414248,0,0,0,0


In [27]:
# Verify the query table's available windows
window_bounds = con.sql(f"""
    SELECT
        MIN(window_start) AS min_window_start,
        MAX(window_end) AS max_window_end,
        COUNT(*) AS rows
    FROM {TABLES['fact_query_90d']}
""").df()

print("Query-window bounds:")
display(window_bounds)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query-window bounds:


,min_window_start,max_window_end,rows
0,2026-04-02,2026-06-30,2414248


In [28]:
# Verify that prev30 and last30 fields actually exist
window_fields = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(*) FILTER (WHERE impressions_prev30 IS NULL)
            AS missing_imp_prev30,
        COUNT(*) FILTER (WHERE impressions_last30 IS NULL)
            AS missing_imp_last30,
        COUNT(*) FILTER (WHERE clicks_prev30 IS NULL)
            AS missing_clicks_prev30,
        COUNT(*) FILTER (WHERE clicks_last30 IS NULL)
            AS missing_clicks_last30
    FROM {TABLES['fact_query_90d']}
""").df()

print("Window-field completeness:")
display(window_fields)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Window-field completeness:


,rows,missing_imp_prev30,missing_imp_last30,missing_clicks_prev30,missing_clicks_last30
0,2414248,0,0,0,0


Verification: The daily performance data runs from 2025-01-27 through 2026-06-30. The query-level table contains 2,414,248 rows covering windows through 2026-06-30. The planned query features have no missing values in this table: content_visible_query_count, rare_query_count, rare_impressions_share, and anonymized_impressions_share are all complete. The query table also provides separate prev30 and last30 fields, with no missing values in the checked impression and click fields.

The feature/label separation is therefore explicit: prev30 fields can support features, while last30 fields are reserved for label construction and must not enter the model feature set.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [29]:
print(con.sql(f"""
    SELECT *
    FROM {TABLES['dim_clients']}
    LIMIT 1
""").df().columns.tolist())

['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']


In [30]:
print(con.sql(f"""
    SELECT
        COUNT(*) AS clients,
        COUNT(*) FILTER (WHERE gsc_data_start IS NULL)
            AS missing_gsc_start,
        COUNT(*) FILTER (WHERE ga4_data_start IS NULL)
            AS missing_ga4_start
    FROM {TABLES['dim_clients']}
""").df())

   clients  missing_gsc_start  missing_ga4_start
0      104                 37                 53


**Missing values / what this data can't tell you:**
- Client data availability: 104 clients are present; 37 have no recorded gsc_data_start and 53 have no recorded ga4_data_start. Because these start dates are missing for some clients, row-level gsc_data_available and ga4_data_available flags should be used when interpreting GSC/GA4 fields rather than assuming that a zero means no activity.

- Rows before a client's `ga4_data_start` have GA4 columns zero-filled with
  `ga4_data_available = FALSE` — must filter on the flag, not treat zeros as "no engagement."
- `fact_content_query_90d` is a fixed 90-day window that overlaps the last30 label period —
  its `*_last30`-style columns are leakage; only `*_prev30`-safe aggregates go in as features.
- Keyword-context missingness (search_volume, competition, cpc) follows `content_type`, not
  randomness — e.g. `feedly article` rows carry no keyword data at all.

**Output:** a ranked list of content items scored by predicted decline risk
(`is_declining_label` probability), handed to a human reviewer as directional
decision-support — never framed as "predicting Google's algorithm."

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.